YAMNet Transfer Learning for Audio Classification Speech Comands
4 reconhecimento de classe: sim, não , silêncio, desconhecido

In [1]:
import tensorflow_hub as hub
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import pathlib
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import random

In [2]:
#fixar as sementes para reprodutibilidade
seed = 42
tf.random.set_seed(seed)#semente para tensoflow
np.random.seed(seed)#semente para numpy
random.seed(seed)#semente para random   
TARGET_CLASS = ["sim", "não", "desconhecido", "silêncio"]
NUM_CLASS = len(TARGET_CLASS)
BATCH_SIZE = 32 #numero de amostras processadas em conjunto em cada etapa
EPOCHS = 20 
LEARNING_RATE = 0.001 #taxa de aprendizado para o Optimizer
SAMPLE_RATE = 16000 #taxa de amostragem dos arquivos de audio
print(f"Classes de destino: {TARGET_CLASS}  Número de classes: {NUM_CLASS}")



Classes de destino: ['sim', 'não', 'desconhecido', 'silêncio']  Número de classes: 4


In [ ]:
import urllib.request
import tarfile
import shutil

def download_and_extract_dataset():
    url = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
    tar_filename = "speech_commands_v0.02.tar.gz"
    extract_dir = "data/datasets"  

    os.makedirs(extract_dir, exist_ok=True)  

    print("Baixando o conjunto de dados...")
    print(f"URL:{url}")
    try:
        urllib.request.urlretrieve(url, tar_filename)
        print(f"Download concluído: {tar_filename}")

        print("Extraindo o conjunto de dados...")
        with tarfile.open(tar_filename, "r:gz") as tar:
            tar.extractall(path=extract_dir)

        os.remove(tar_filename)
        print("Arquivo temporariamente removido.")

        return pathlib.Path(extract_dir) / "speech_commands_v0.02"  # ✅ mudou aqui

    except Exception as e:
        print(f"Erro ao baixar ou extrair o conjunto de dados: {e}")
        return None


possible_dirs = [
    pathlib.Path("data"), 
    pathlib.Path("data/speech_commands_v0.02"),
    pathlib.Path("data/speech_commands_v0_extracted"),
    pathlib.Path("data/datasets/speech_commands_v0.02"),
    pathlib.Path("data/datasets/speech_commands_v0_extracted"),
    pathlib.Path("speech_commands_v0_extracted"),
]


data_dir = None

# -------- procura primeiro --------
for dir_path in possible_dirs:
    if dir_path.exists():
        subdirs = [d.name for d in dir_path.iterdir() if d.is_dir()]
        if len(subdirs) > 0:
            print(f"Conjunto de dados encontrados em: {dir_path}")
            data_dir = dir_path
            break

# -------- tenta keras (uma vez só) --------
if data_dir is None:
    print("Conjunto de dados não encontrado. Tentando baixar com tf.keras...")
    try:
        dataset_path = tf.keras.utils.get_file(
            fname='speech_commands_v0.02.tar.gz',
            origin='http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz',
            extract=True,
            cache_dir='data',
        )

        # ===== DEBUG — AQUI =====
        print("dataset_path retornado =", dataset_path)

        p = pathlib.Path(dataset_path)
        print("parent =", p.parent)

        print("conteúdo da pasta parent:")
        for x in list(p.parent.iterdir())[:20]:
            print(" -", x.name)
        # ===== FIM DEBUG =====

        dataset_dir = pathlib.Path(dataset_path)  

        if dataset_dir.exists():
            data_dir = dataset_dir
            print(f"Conjunto de dados baixado e encontrado em: {data_dir}")

    except Exception as e:
        print(f"Erro como baixar o conjunto de dados com tf.keras: {e}")

# -------- fallback manual --------
if data_dir is None:
    print("Tentando fazer o download manualmente...")
    data_dir = download_and_extract_dataset()
    if data_dir is not None and data_dir.exists():
        print(f"Conjunto de dados baixado manualmente e encontrado em: {data_dir}")

print("data_dir final =", data_dir)


Conjunto de dados encontrados em: data
data_dir final = data


In [3]:

# RECURSOS DE PRE PROCESSAMENTO DE AUDIO
def load_wav_16k_mono(filename):
    """
    Carrega um arquivo wav e o converte para 16khz.
    Esse recurso padroniza todos os áudios para YAMNet.

    Args:
        filename: caminho para o arquivo de audio a ser carregado
    Returns:
        wav: Tensor 1D float32 com exatamente 16000 amostras
    """
    # lê o conteúdo binário do arquivo de áudio
    file_contents = tf.io.read_file(filename)

    # decodifica WAV (força mono)
    wav, sample_rate = tf.audio.decode_wav(file_contents, desired_channels=1)

    # remove dimensão do canal (fica [samples])
    wav = tf.squeeze(wav, axis=-1)

    # converte taxa de amostragem para int64 para comparações
    sample_rate = tf.cast(sample_rate, dtype=tf.int64)

    # reamostragem para 16khz, se necessário
    if tf.not_equal(sample_rate, 16000):
        resample_factor = tf.cast(sample_rate, tf.float32) / 16000.0

        # novo tamanho aproximado após reamostrar
        new_length = tf.cast(
            tf.cast(tf.shape(wav)[0], tf.float32) / resample_factor,
            tf.int32
        )

        # índices (linear) para "simular" reamostragem
        idx = tf.cast(
            tf.linspace(
                0.0,
                tf.cast(tf.shape(wav)[0] - 1, tf.float32),
                new_length
            ),
            tf.int32
        )
        wav = tf.gather(wav, idx)

    # garante float32
    wav = tf.cast(wav, tf.float32)

    # padronizar comprimento para exatamente 1 segundo (16000 amostras)
    target_length = 16000
    current_length = tf.shape(wav)[0]

    if tf.less(current_length, target_length):
        padding = target_length - current_length
        wav = tf.pad(wav, [[0, padding]], mode="CONSTANT")
    else:
        wav = wav[:target_length]

    return wav

def preprocess_dataser(files, labels):
    """
    Pr-processamento completo do conjuntos de dados: carrega áudio e prepara os rótulos.
    Cria uma tf.data.Dataset otimizado treinado
    Args:
    files: lista de caminhos de arquivos de audio
    labels: lista de rotulos numericos correspondentes
    return: conjunto de dados tf.data.Dataset de dados pré-processados com audio e rotulos
    """
    #criar um conjunto de dados a partir de caminho de arquivos
    path_ds = tf.data.Dataset.from_tensor_slices(files)
    #Aplique o recurso de upload de audioi a todas as rotas
    #num_parallel_calls = AUTOTUNE  otimiza automaticamente o paralelismo
    audio_ds = path_ds.map(load_wav_16k_mono, num_parallel_calls = tf.data.AUTOTUNE)
    #criar um conjunto de dados a partir de rótulos
    labels_ds = tf.data.Dataset.from_tensor_slices(labels)
    #combinar os audios e as etiquetas num único conjunto de dados
    #zip criar pares (áudio, etiqueta)
    return tf.data.Dataset.zip((audio_ds, labels_ds))


In [ ]:
#PREPARAR DADOS PARA TREINAMENTO
#pre verificar se o conjunto de dados existe
if data_dir is None or not data_dir.exists():
    print(f"ERROR: O diretório {data_dir} não existe. Verifique o caminho e tente novamente.")
else:
    #listar todos os itens do diretorio
    all_items = list(data_dir.iterdir())
    #separar diretorio(classes) de arquivo
    diretorios = [item for item in all_items if item.is_dir()]
    files = [item for item in all_items if item.is_file()]
    print(f"Diretorios encontrados:{len(diretorios)}  Arquivos encontrados: {len(files)}")
#verificação de integridade: ve se tem diretorios suficientes
#o conjunto deve ter cercade 35 classes
if len(diretorios) < 10:
    print(f"WARNING: Você pode encontrar alguns diretorios. Possivel problema de  extracao.")
    print(f"Diretorios disponiveis:")
    for d in diretorios:
        print(f" - {d.name}")
    
    for d in diretorios:
        #Verificar se este diretorio contem muitos subdiretorios
        subdirs = [item for item in d.iterdir() if item.is_dir()]
        print(f"Conjunto de dados provavelmente em :{d}")
        data_dir  = d
        break
    #obtem os nome sde todas as classes disponiveis
    comands =[d.name for d in data_dir.iterdir() if d.is_dir()]
    #remove todos os arquivos README.md da lista de classes
    comands = [cmd for cmd in comands if cmd != "README.md"]
    print(f"Diretorio final do conjunto de dados: {data_dir}")
    print(f"Classes disponiveis: {comands}")
    print(f"Primeiras 10 classess: {comands[:10] if len(comands) > 10 else comands}")
    #inicialização de listas para coletare todos os arfquivos e rotulos
    all_files = [] #lista de todos os caminhos de arquivo de audio
    all_labels = [] #lista de todos os rótulos numéricos correspondentes
    
    #Coleta de arquivos para classes 'sim e 'não'
    for target_class in ["sim", "não"]:
        #verifique se a classe existe no conjunto
        if target_class in comands:
            #construa o caminho para o diretorio classe
            class_dir = data_dir / target_class
            if class_dir.exists():
                #encontras todos os carquivos .wav no diretorio de classe
                files = list(class_dir.glob("*.wav"))
                #converte objetos Path em string  para compatibilidade
                files = [str(f) for f in files]
                #Adiciona os rotulos correspondente(repetidos para cadada arquivo)
                #adiciona os arquivos  a lista geral
                all_files.extend(files)
                #adiciona os rotulos correspondentes 
                all_labels.extend([target_class] * len(files))
                print(f" Classe '{target_class}': {len(files)} arquivo.")
            else:
                print(f"WARNING: O diretorio '{class_dir}' não existe")
        else:
            print(f"AVISO: A classe '{target_class}' não foi encontrada no conjunto de dados.")

        #coleta de arquivos para  classe 'silencio', ruido de fundo
        silence_dir = data_dir /"_background_noise_"
        if silence_dir.exists() and "_background_noise_" in comands:
            #encontre todos os arquivos de audio no diretorio de ruido de fundo
            silence_files = list(silence_dir.glob("*.wav"))
            silence_files = [str(f) for f in silence_files]
            #Limite o numero de arquivos de silencio para equilibrar o conjunto de dados
            #muitos arquivos de ruido de fundo podem desequilibrar o treinamento
            silence_files = silence_files[:1000] if len(silence_files)>1000 else silence_files
            all_files.extend(silence_files)
            


            
